## dependencies

In [10]:
from dotenv import load_dotenv
load_dotenv()

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

import os
from datetime import datetime
import pytz
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor, tool
from typing import Dict, Any
from pydantic import BaseModel, Field

## global config

In [11]:
LLM_MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.1

## class and tool for SystemTimeInput

In [12]:
class SystemTimeInput(BaseModel):
    format_str: str = Field(
        default="%Y-%m-%d %H:%M:%S",
        description="The format in which to return the current system time."
    )

In [13]:
@tool("get_system_time", args_schema=SystemTimeInput)
def get_system_time(format_str: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    Get the current system time in the specified format.
    
    Args:
        format_str: Time format string (default: "%Y-%m-%d %H:%M:%S")
        
    Returns:
        Formatted current time string
    """
    try:
        current_time = datetime.now().strftime(format_str)
        logger.info(f"Current time: {current_time}")
        return current_time
    except Exception as e:
        logger.error(f"❌ Error getting current time: {e}")
        return f"Error getting current time: {e}"

## class and tool for TimezoneTimeInput

In [14]:
class TimezoneTimeInput(BaseModel):
    timezone: str = Field(description="Timezone name (e.g., 'Europe/London', 'Asia/Kolkata', 'mumbai', 'london')")
    format_str: str = Field(default="%H:%M:%S", description="Time format string")

In [15]:
@tool("get_timezone_time", args_schema=TimezoneTimeInput)
def get_timezone_time(timezone: str, format_str: str = "%H:%M:%S") -> str:
    """
    Get the current time in a specific timezone.
    
    Args:
        timezone: Timezone name (e.g., 'Europe/London', 'Asia/Kolkata', 'mumbai', 'london')
        format_str: Time format string (default: "%H:%M:%S")
        
    Returns:
        Formatted time string for the specified timezone
    """
    try:
        # Handle case where timezone might be passed as JSON string
        if timezone.startswith('{"') and timezone.endswith('"}'):
            import json
            parsed_args = json.loads(timezone)
            timezone = parsed_args.get('timezone', timezone)
            format_str = parsed_args.get('format_str', format_str)
            
        # Common timezone mappings
        timezone_map = {
            'london': 'Europe/London',
            'mumbai': 'Asia/Kolkata',
        }
        
        # Normalize timezone input
        tz_name = timezone_map.get(timezone.lower(), timezone)
        
        # Get timezone object
        tz = pytz.timezone(tz_name)
        
        # Get current time in timezone
        current_time = datetime.now(tz)
        formatted_time = current_time.strftime(format_str)
        
        logger.info(f"Retrieved time for {timezone}: {formatted_time}")
        return f"Current time in {timezone}: {formatted_time}"
        
    except pytz.exceptions.UnknownTimeZoneError:
        error_msg = f"Unknown timezone: {timezone}"
        logger.error(error_msg)
        return error_msg
    except Exception as e:
        error_msg = f"Error getting timezone time: {str(e)}"
        logger.error(error_msg)
        return error_msg

## class and functions for TimeAgentSystem

In [16]:
class TimeAgentSystem:
    def __init__(
            self,
            model: str = LLM_MODEL,
            temperature: float = TEMPERATURE,
            verbose: bool = True,
    ):
        self.model = model
        self.temperature = temperature
        self.verbose = verbose

        self._validate_env()

        self._setup_agent()

    def _validate_env(self) -> None:
        if not os.getenv("GOOGLE_API_KEY"):
            raise ValueError("❌ GOOGLE_API_KEY environment variable is not set.")
        
    def _setup_agent(self) -> None:
        try:
            # Initialize LLM
            llm = ChatGoogleGenerativeAI(
                model=self.model,
                temperature=self.temperature
            )
            
            # Load ReAct prompt template
            prompt_template = hub.pull("hwchase17/react")
            
            # Define tools
            tools = [get_system_time, get_timezone_time]
            
            # Create agent
            agent = create_react_agent(llm, tools, prompt_template)
            
            # Create agent executor
            self.agent_executor = AgentExecutor(
                agent=agent,
                tools=tools,
                verbose=self.verbose,
                handle_parsing_errors=True,
                max_iterations=3,
            )

            logger.info("✅ Agent setup successfully.")

        except Exception as e:
            logger.error(f"❌ Error setting up agent: {e}")
            raise

    def ask_question(self, query: str) -> Dict[str, Any]:
        """
        Process a query using the agent system.
        
        Args:
            query: The question to ask the agent
            
        Returns:
            Dictionary containing the response and metadata
        """
        if not self.agent_executor:
            raise RuntimeError("Agent executor is not set up. Call _setup_agent() first.")
        
        try:
            logger.info(f"Processing query: {query}")

            result = self.agent_executor.invoke({"input": query})

            response = {
                "query": query,
                "output": result.get("output", "No response generated"),
                "success": True
            }

            logger.info("Query processed successfully")
            
            return response
        
        except Exception as e:
            logger.error(f"❌ Error processing query: {e}")
            return {
                "query": query,
                "output": str(e),
                "success": False
            }

## main function to run the agent

In [17]:
def main():
    try:
        print("🚀 Initializing Time Agent System...")
        agent_system = TimeAgentSystem()

        # Sample queries
        sample_queries = [
            "What is the current time in Mumbai? Just show the time, not the date.",
            "What time is it in London right now?",
            "Compare the current time in London and Mumbai."
        ]

        print("\n📝 Running sample queries...")
        for query in sample_queries:
            print('=' * 50)
            print(f"\n❓ Query: {query}")
            response = agent_system.ask_question(query)
            print(f"🤖 Response: {response['output']}")
    except Exception as e:
        logger.error(f"❌ Error in main function: {e}")
        raise

In [18]:
main()

🚀 Initializing Time Agent System...


/Users/hi/jitenStuff/MyGit/Artificial-Intelligence/.venv/lib/python3.13/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
2025-08-04 14:43:12,304 - __main__ - INFO - ✅ Agent setup successfully.
2025-08-04 14:43:12,305 - __main__ - INFO - Processing query: What is the current time in Mumbai? Just show the time, not the date.



📝 Running sample queries...

❓ Query: What is the current time in Mumbai? Just show the time, not the date.


> Entering new AgentExecutor chain...


2025-08-04 14:43:13,605 - __main__ - INFO - Retrieved time for mumbai: 14:43:13


Action: get_timezone_time
Action Input: {"timezone": "mumbai", "format_str": "%H:%M:%S"}Current time in mumbai: 14:43:13

...

Action: get_timezone_time
Action Input: {"timezone": "Europe/London"}Current time in Europe/London: 10:13:18

2025-08-04 14:43:19,682 - __main__ - INFO - Retrieved time for Asia/Kolkata: 14:43:19


Action: get_timezone_time
Action Input: {"timezone": "Asia/Kolkata"}Current time in Asia/Kolkata: 14:43:19

2025-08-04 14:43:20,801 - __main__ - INFO - Query processed successfully


I now know the final answer
Final Answer: The current time in London is 10:13:18 and in Mumbai it is 14:43:19.

> Finished chain.
🤖 Response: The current time in London is 10:13:18 and in Mumbai it is 14:43:19.
